# MExMFA qMRI synthetic-data demo\nThis notebook contains no human data. It runs a small 20-condition (5 FA × 4 TE) joint-reconstruction and qMRI-fit demonstration using the compiled public module.

In [ ]:
from pathlib import Path\nimport csv\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom mexmfa_demo.core import DemoConfig, make_synthetic_dataset\nfrom mexmfa_zssl import run_production_zssl\n\nROOT = Path.cwd()\nDATA = ROOT / 'data' / 'synthetic_af4_zssl.npz'\nOUTPUT = ROOT / 'results'\nRUN_ANALYSIS = False  # Set True to run 2 epochs of production ZS-SSL.\nOVERWRITE_OUTPUT = False  # Set True only to replace files in results/.\n

In [ ]:
if RUN_ANALYSIS:\n    if OUTPUT.exists() and any(OUTPUT.iterdir()) and not OVERWRITE_OUTPUT:\n        raise RuntimeError('results/ exists. Set OVERWRITE_OUTPUT=True to replace it.')\n    if OVERWRITE_OUTPUT:\n        for item in OUTPUT.glob('*'):\n            if item.is_file(): item.unlink()\n    make_synthetic_dataset(DATA, DemoConfig(matrix=24))\n    result = run_production_zssl(DATA, OUTPUT, epochs=2, seed=100, device='cpu')\n    print('Wrote:', OUTPUT, 'loss:', result['loss_csv'])\nelse:\n    print('Review mode. Set RUN_ANALYSIS=True, then run this cell.')

In [ ]:
metrics_path = OUTPUT / 'image_metrics_zssl.csv'\nloss_path = OUTPUT / 'loss_zssl.csv'\nif metrics_path.exists() and loss_path.exists():\n    with metrics_path.open() as f: metrics = list(csv.DictReader(f))\n    for key in ('nrmse', 'psnr_db', 'ssim'):\n        print(f'mean {key}: {np.mean([float(m[key]) for m in metrics]):.4f}')\n    print('loss records:')\n    print(loss_path.read_text())\nelse:\n    print('No results yet.')

In [ ]:
if (OUTPUT / 'T1_s_zssl.npy').exists():\n    maps = [(np.load(OUTPUT/'M0_zssl.npy'), 'M0', 'gray'), (np.load(OUTPUT/'T1_s_zssl.npy'), 'T1 [s]', 'magma'), (np.load(OUTPUT/'T2star_s_zssl.npy'), 'T2* [s]', 'magma')]\n    fig, axes = plt.subplots(1, 3, figsize=(12, 4))\n    for ax, (image, title, cmap) in zip(axes, maps):\n        h = ax.imshow(image.T, cmap=cmap); ax.set_title(title); ax.axis('off'); fig.colorbar(h, ax=ax, shrink=.7)\n    plt.show()

## Readiness\nThe compiled module is CPython 3.11-specific and is validated through the Docker image. Outputs are functional synthetic-data checks, not paper results. See `README.md` for the public scope and withheld real-data requirements.